[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinod-seth/Applied-Scientist-Interview-Gauntlet/blob/main/tutorial/02_systems_and_evaluation/02_calibration_and_temperature_lab.ipynb)

# Armory Notebook 5 — Calibration & Temperature Lab

| | |
|---|---|
| **Companion to** | Session 2, Lesson 2 — Defending Calibration Under Distribution Shift |
| **Runtime** | CPU only (synthetic logits; no model downloads) |
| **Estimated time** | 30 minutes |
| **Last verified** | 2026-07-14 |

Every claim in Lesson 2 becomes executable here: ECE from scratch (both binning schemes), reliability diagrams, temperature fitting by NLL, the accuracy-invariance proof as an assertion, and — the centrepiece — the **oracle-temperature sweep** that shows *why* a clean-fitted T cannot follow distribution shift.

Logits are simulated with a shift model whose severity dial you control. That is the point: with known ground truth you can verify your estimator recovers it, and you can see the mechanism isolated from the noise of a real corruption benchmark.

> [!IMPORTANT]
> Synthetic. Nothing here enters the Metric Vault — your ECE, your fitted T, and your per-severity curves come from your own runs only.

In [ ]:
%pip install -q "numpy>=1.26" "scipy>=1.11" "matplotlib>=3.8"

## Part 1 — A shift model that reproduces the phenomenon

The generator below encodes Lesson 2's mechanism directly:

- **True class separation shrinks** with severity → accuracy falls (features genuinely degrade).
- **Logit magnitude does *not* shrink proportionally** → the linear head keeps producing large margins on off-manifold features → softmax stays saturated → confidence persists.

That decoupling is the whole finding, expressed in two lines of simulation.

In [ ]:
import numpy as np

N_CLASSES = 10
rng = np.random.default_rng(7)

def make_logits(n, severity, rng):
    """Simulate a network's logits at a given corruption severity (0 = clean).

    Two knobs, and their DIFFERENT decay rates are the whole phenomenon:

    signal : true-class advantage. Decays fast -> accuracy collapses.
             Only signal affects argmax, so it alone sets accuracy.
    scale  : overall logit magnitude. Decays slowly -> softmax stays saturated
             -> confidence persists. Scale never changes argmax, only sharpness.

    Calibrated at severity 0 to mirror a real modern network: ~0.91 accuracy,
    ~0.96 mean confidence, i.e. OVERconfident, needing T ~ 2 (the range Guo
    et al. 2017 report for ResNets).
    """
    signal = 3.0 * np.exp(-0.42 * severity)     # discriminative information
    scale  = 6.0 * np.exp(-0.05 * severity)     # confidence-producing margin
    labels = rng.integers(0, N_CLASSES, size=n)
    logits = rng.normal(0, 1.0, size=(n, N_CLASSES))
    logits[np.arange(n), labels] += signal
    return logits * scale, labels

def softmax(z, T=1.0):
    z = z / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

for sev in (0, 2, 5):
    lg, lb = make_logits(4000, sev, np.random.default_rng(sev))
    p = softmax(lg)
    acc = (p.argmax(1) == lb).mean()
    conf = p.max(1).mean()
    print(f"severity {sev}:  accuracy={acc:.3f}   mean confidence={conf:.3f}   gap={conf-acc:+.3f}")
print("\nAt severity 0 the gap is small and positive: mildly overconfident, like a real net.")
print("As severity rises, accuracy collapses while confidence barely moves - the gap")
print("widens into the resume claim. Note the model has NO mechanism to notice it left")
print("familiar territory: nothing in a discriminative head represents 'this input is odd'.")

## Part 2 — ECE from scratch, both binning schemes

Lesson 2, Refresher 1. The same predictions scored two ways — and the disagreement is the lesson.

In [ ]:
def ece(probs, labels, n_bins=15, scheme="fixed"):
    """Top-label ECE. scheme='fixed' (equal-width) or 'mass' (equal-count)."""
    conf = probs.max(axis=1)
    correct = (probs.argmax(axis=1) == labels).astype(float)
    n = len(conf)

    if scheme == "fixed":
        edges = np.linspace(0.0, 1.0, n_bins + 1)
    else:                                     # equal-mass: quantile edges
        edges = np.quantile(conf, np.linspace(0, 1, n_bins + 1))
        edges[0], edges[-1] = 0.0, 1.0

    total, rows = 0.0, []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (conf > lo) & (conf <= hi) if lo > 0 else (conf >= lo) & (conf <= hi)
        if sel.sum() == 0:
            rows.append((lo, hi, 0, np.nan, np.nan)); continue
        bin_conf, bin_acc = conf[sel].mean(), correct[sel].mean()
        total += (sel.sum() / n) * abs(bin_acc - bin_conf)
        rows.append((lo, hi, int(sel.sum()), bin_conf, bin_acc))
    return total, rows

logits_clean, labels_clean = make_logits(6000, severity=0, rng=np.random.default_rng(1))
probs_clean = softmax(logits_clean)

e_fixed, rows_fixed = ece(probs_clean, labels_clean, scheme="fixed")
e_mass,  rows_mass  = ece(probs_clean, labels_clean, scheme="mass")
print(f"ECE (15 fixed-width bins): {e_fixed:.4f}")
print(f"ECE (15 equal-mass bins) : {e_mass:.4f}")

occupied = sum(1 for r in rows_fixed if r[2] > 0)
print(f"\nfixed-width bins actually populated: {occupied}/15")
top_mass = sum(r[2] for r in rows_fixed if r[0] >= 0.9) / len(labels_clean)
print(f"prediction mass above confidence 0.9: {top_mass:.1%}")
print("\nThat concentration is why the two schemes disagree - and why quoting ECE")
print("without its binning scheme is not a reproducible claim.")

## Part 3 — Reliability diagram, read like an instrument

With the bin-population histogram attached — because a dramatic gap in a near-empty bin is noise, not a finding.

In [ ]:
import matplotlib.pyplot as plt

INK, INK_SOFT = "#0b0b0b", "#52514e"
BLUE, AQUA, RED, VIOLET = "#2a78d6", "#1baf7a", "#e34948", "#4a3aa7"

def reliability_plot(ax_top, ax_bot, probs, labels, title):
    _, rows = ece(probs, labels, n_bins=15, scheme="fixed")
    centers = [(lo + hi) / 2 for lo, hi, *_ in rows]
    accs    = [r[4] for r in rows]
    counts  = [r[2] for r in rows]

    ax_top.plot([0, 1], [0, 1], linestyle="--", color=INK_SOFT, linewidth=1, label="perfect calibration")
    ax_top.bar(centers, accs, width=1/15 * 0.9, color=BLUE, edgecolor="#fcfcfb", linewidth=0.5, label="accuracy")
    ax_top.set_ylabel("accuracy", color=INK_SOFT, fontsize=9)
    ax_top.set_xlim(0, 1); ax_top.set_ylim(0, 1)
    ax_top.set_title(title, fontsize=10, color=INK)
    ax_top.legend(fontsize=7, frameon=False, loc="upper left")
    ax_top.tick_params(colors=INK_SOFT, labelsize=8)
    ax_top.spines[["top", "right"]].set_visible(False)

    ax_bot.bar(centers, counts, width=1/15 * 0.9, color=AQUA, edgecolor="#fcfcfb", linewidth=0.5)
    ax_bot.set_xlabel("confidence", color=INK_SOFT, fontsize=9)
    ax_bot.set_ylabel("count", color=INK_SOFT, fontsize=9)
    ax_bot.set_xlim(0, 1)
    ax_bot.tick_params(colors=INK_SOFT, labelsize=8)
    ax_bot.spines[["top", "right"]].set_visible(False)

logits_sev4, labels_sev4 = make_logits(6000, severity=4, rng=np.random.default_rng(4))
probs_sev4 = softmax(logits_sev4)

fig, axes = plt.subplots(2, 2, figsize=(11, 5.4), height_ratios=[3, 1])
reliability_plot(axes[0][0], axes[1][0], probs_clean, labels_clean, "Clean (severity 0)")
reliability_plot(axes[0][1], axes[1][1], probs_sev4, labels_sev4, "Shifted (severity 4)")
plt.tight_layout(); plt.show()

print(f"clean   ECE: {ece(probs_clean, labels_clean)[0]:.4f}")
print(f"sev-4   ECE: {ece(probs_sev4, labels_sev4)[0]:.4f}")
print("\nBars sagging further below the diagonal = overconfidence widening under shift.")

## Part 4 — Fit T by NLL, and assert accuracy invariance

Lesson 2, Refresher 2. Two claims made executable: NLL is smooth and convex in T (so a 1-D optimizer finds it trivially), and accuracy is **bit-for-bit unchanged** — the free integrity check on any TS implementation.

In [ ]:
from scipy.optimize import minimize_scalar

def nll(logits, labels, T):
    p = softmax(logits, T)
    return -np.log(p[np.arange(len(labels)), labels] + 1e-12).mean()

def fit_temperature(logits, labels):
    # Bounds must be wide: under heavy shift the optimal T climbs far above 1.
    # A cap that binds would silently fake the very result we are testing.
    res = minimize_scalar(lambda t: nll(logits, labels, t), bounds=(0.05, 25.0), method="bounded")
    return float(res.x)

T_clean = fit_temperature(logits_clean, labels_clean)
print(f"T fitted on CLEAN validation: {T_clean:.3f}")
print("  T > 1 => the raw model was overconfident and TS is softening it.")
print("  (Guo et al. 2017 report T ~ 1.5-3 for ResNet-class models: we are in range.)\n")

# --- accuracy invariance: not an opinion, an assertion ---
acc_before = (softmax(logits_clean).argmax(1) == labels_clean).mean()
acc_after  = (softmax(logits_clean, T_clean).argmax(1) == labels_clean).mean()
assert acc_before == acc_after, "TS changed accuracy -> implementation bug"
print(f"accuracy before TS: {acc_before:.4f}")
print(f"accuracy after  TS: {acc_after:.4f}   <- identical, as monotonicity requires")
print("  This assert is a free integrity check on ANY temperature-scaling code:")
print("  if accuracy moves, you refit the head or applied T inconsistently.\n")

print(f"clean ECE before TS: {ece(softmax(logits_clean), labels_clean)[0]:.4f}")
print(f"clean ECE after  TS: {ece(softmax(logits_clean, T_clean), labels_clean)[0]:.4f}")
print("\nIn-distribution, one scalar does the job. Now break it.")

In [ ]:
# Show the NLL curve is smooth/convex in T while ECE is a step function - Lesson 2 Follow-up 3.
T_grid = np.linspace(0.4, 6.0, 200)
nll_curve = [nll(logits_clean, labels_clean, t) for t in T_grid]
ece_curve = [ece(softmax(logits_clean, t), labels_clean)[0] for t in T_grid]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(T_grid, nll_curve, color=BLUE, linewidth=2)
ax1.axvline(T_clean, color=RED, linestyle="--", linewidth=1.5, label=f"fitted T={T_clean:.2f}")
ax1.set_title("NLL vs T — smooth, convex, optimizable", fontsize=10, color=INK)
ax2.plot(T_grid, ece_curve, color=VIOLET, linewidth=2)
ax2.set_title("ECE vs T — piecewise-constant, hostile to gradients", fontsize=10, color=INK)
for ax, yl in ((ax1, "NLL"), (ax2, "ECE")):
    ax.set_xlabel("temperature T", color=INK_SOFT, fontsize=9)
    ax.set_ylabel(yl, color=INK_SOFT, fontsize=9)
    ax.tick_params(colors=INK_SOFT, labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.7); ax.set_axisbelow(True)
ax1.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

print("This picture IS the answer to 'why NLL and not ECE as the objective?'")
print("Zoom into the right panel: ECE moves in flat steps because examples cross bin")
print("edges only at discrete T values. Gradient-based optimization has nothing to")
print("descend. NLL on the left is convex in 1/T and lands the optimum in one call.")

## Part 5 — The transfer failure, and the oracle-T exhibit

The centrepiece (Lesson 2, Follow-up 4). Apply the **clean-fitted** T across severities, and separately refit an **oracle** T at each severity. If oracle-T climbs, the *needed* correction is moving — proving no single scalar can serve the family. This is the difference between demonstrating the symptom and demonstrating the mechanism.

In [ ]:
severities = [0, 1, 2, 3, 4, 5]
results = []
for sev in severities:
    lg, lb = make_logits(6000, sev, np.random.default_rng(100 + sev))
    acc = (softmax(lg).argmax(1) == lb).mean()
    results.append({
        "severity": sev,
        "accuracy": acc,
        "confidence": softmax(lg).max(1).mean(),
        "ece_raw":  ece(softmax(lg), lb)[0],
        "ece_cleanT": ece(softmax(lg, T_clean), lb)[0],
        "T_oracle": fit_temperature(lg, lb),
    })
    results[-1]["ece_oracleT"] = ece(softmax(lg, results[-1]["T_oracle"]), lb)[0]

print(f"{'sev':>3} {'acc':>6} {'conf':>6} {'ECE raw':>8} {'ECE cleanT':>11} {'ECE oracleT':>12} {'T*':>6}")
for r in results:
    print(f"{r['severity']:>3} {r['accuracy']:>6.3f} {r['confidence']:>6.3f} "
          f"{r['ece_raw']:>8.4f} {r['ece_cleanT']:>11.4f} {r['ece_oracleT']:>12.4f} {r['T_oracle']:>6.2f}")
print(f"\nclean-fitted T = {T_clean:.2f} (constant, applied everywhere)")
print("T* = oracle temperature refit at each severity.")

In [ ]:
sev = [r["severity"] for r in results]
fig, (axA, axB) = plt.subplots(1, 2, figsize=(11, 3.9))

axA.plot(sev, [r["accuracy"] for r in results], marker="o", color=AQUA, linewidth=2, label="accuracy")
axA.plot(sev, [r["confidence"] for r in results], marker="s", color=RED, linewidth=2, label="mean confidence")
axA.set_title("Confidence persists while accuracy collapses", fontsize=10, color=INK)
axA.set_ylabel("rate", color=INK_SOFT, fontsize=9); axA.set_ylim(0, 1.05)
axA.legend(fontsize=8, frameon=False)

axB.plot(sev, [r["ece_cleanT"] for r in results], marker="s", color=RED, linewidth=2,
         label=f"ECE with clean-fitted T={T_clean:.2f}")
axB.plot(sev, [r["ece_oracleT"] for r in results], marker="^", color=BLUE, linewidth=2,
         label="ECE with oracle T per severity")
axB.set_title("Clean-fitted T stops working; a per-severity T doesn't", fontsize=10, color=INK)
axB.set_ylabel("ECE", color=INK_SOFT, fontsize=9)
axB.legend(fontsize=8, frameon=False)

for ax in (axA, axB):
    ax.set_xlabel("corruption severity", color=INK_SOFT, fontsize=9)
    ax.tick_params(colors=INK_SOFT, labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e6e5e0", linewidth=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(sev, [r["T_oracle"] for r in results], marker="o", color=VIOLET, linewidth=2, label="oracle T*(severity)")
ax.axhline(T_clean, color=RED, linestyle="--", linewidth=1.5, label=f"clean-fitted T={T_clean:.2f}")
ax.set_xlabel("corruption severity", color=INK_SOFT, fontsize=9)
ax.set_ylabel("temperature", color=INK_SOFT, fontsize=9)
ax.set_title("THE EXHIBIT: the needed correction moves with severity", fontsize=10, color=INK)
ax.legend(fontsize=8, frameon=False)
ax.tick_params(colors=INK_SOFT, labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e6e5e0", linewidth=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

print("Say this out loud until it is reflex:")
print("  Temperature scaling assumes miscalibration is ONE global logit-scale factor.")
print("  Under shift, off-manifold features meet a linear head that extrapolates confidently,")
print("  so the required correction grows with severity. A scalar fitted on clean data is")
print("  simply the wrong member of a family - which is what the rising T* curve proves.")

## Part 6 — Acting under miscalibration: the risk–coverage curve

Lesson 2, Follow-up 6 and Session 2's bridge to the RAG project's abstention story. Selective prediction converts unreliable answers into visible refusals — and the honest reporting format is a curve, never a single accuracy number.

In [ ]:
def risk_coverage(probs, labels):
    conf = probs.max(1)
    correct = (probs.argmax(1) == labels).astype(float)
    order = np.argsort(-conf)                 # answer most-confident first
    correct_sorted = correct[order]
    coverage = np.arange(1, len(conf) + 1) / len(conf)
    risk = 1 - np.cumsum(correct_sorted) / np.arange(1, len(conf) + 1)
    return coverage, risk

fig, ax = plt.subplots(figsize=(6.8, 3.8))
for sev, color in ((0, BLUE), (3, VIOLET), (5, RED)):
    lg, lb = make_logits(6000, sev, np.random.default_rng(200 + sev))
    cov, risk = risk_coverage(softmax(lg, T_clean), lb)
    ax.plot(cov, risk, color=color, linewidth=2, label=f"severity {sev}")
ax.set_xlabel("coverage (fraction answered)", color=INK_SOFT, fontsize=9)
ax.set_ylabel("risk (error rate among answered)", color=INK_SOFT, fontsize=9)
ax.set_title("Risk–coverage: abstention buys correctness, and shift raises the price",
             fontsize=10, color=INK)
ax.legend(fontsize=8, frameon=False)
ax.tick_params(colors=INK_SOFT, labelsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(color="#e6e5e0", linewidth=0.7); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

print("Note what the curves do NOT show: any severity-5 operating point matching")
print("severity-0 risk at useful coverage. Selective prediction controls risk;")
print("it does not restore the accuracy shift destroyed. That non-promise is the")
print("sentence a Bar Raiser is listening for.")

## Close the loop

1. **Diff against your project.** Which binning scheme did you report, and did you ever compute the other? Did you fit an oracle T per severity? Every gap is a gap for [PROGRESS.md](../../PROGRESS.md).
2. **Fill the vault from your runs only** — architecture, dataset, per-severity accuracy/ECE, fitted T, oracle-T if run. Empty slots stay `QUALITATIVE-ONLY`.
3. **assessment rehearsal (Follow-up 7).** Looking at the oracle-T plot, explain in 60 seconds why clean-fitted T fails at severity 4 — mechanism, not observation. If the word "because" appears fewer than twice, it isn't a mechanism yet.
4. **The honesty rep (Follow-up 8).** Answer out loud: "This is a known result — what did *you* add?" The lab you just ran is part of the true answer: validated instruments, cross-checked estimators, and a mechanism exhibit rather than a citation.

## References & Credits

- Guo et al. (2017), *On Calibration of Modern Neural Networks* — https://arxiv.org/abs/1706.04599 (binning ECE estimator originally from Naeini, Cooper & Hauskrecht, AAAI 2015)
- Hendrycks & Dietterich (2019), *Benchmarking Neural Network Robustness to Common Corruptions and Perturbations* — https://arxiv.org/abs/1903.12261
- Ovadia et al. (2019), *Can You Trust Your Model's Uncertainty?* — https://arxiv.org/abs/1906.02530
- Lakshminarayanan et al. (2017), *Deep Ensembles* — https://arxiv.org/abs/1612.01474
- NumPy (BSD-3), SciPy (BSD-3), Matplotlib (PSF-based license).